<a href="https://colab.research.google.com/github/illelias/CLIP/blob/main/CLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch pillow

In [ ]:
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
import itertools

# Load model
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Load image
image = Image.open("/content/me_dog.jpg").convert("RGB")

# -------------------------
# DEFINE COMPONENTS
# -------------------------

genders = ["man", "woman"]

races = ["white", "black", "asian", "latino"]

jobs = ["civilian", "soldier", "police officer", "firefighter", "businessman"]

animals = [
    "",
    "standing with a dog",
    "standing with a corgi"
]

trees = [
    "",
    "under a tree",
    "under a cherry blossom tree",
    "under a palm tree",
    "under an oak tree"
]

# -------------------------
# GENERATE DESCRIPTIONS
# -------------------------

descriptions = []

for gender, race, job, animal, tree in itertools.product(
    genders, races, jobs, animals, trees
):
    sentence = f"a photo of a {race} {job} {gender}"

    if animal:
        sentence += f" {animal}"
    if tree:
        sentence += f" {tree}"

    descriptions.append(sentence)

# Remove duplicates just in case
descriptions = list(set(descriptions))

print(f"Total descriptions: {len(descriptions)}")
print(descriptions[:10])  # preview

In [ ]:
inputs = processor(
    text=descriptions,
    images=image,
    return_tensors="pt",
    padding=True
)

with torch.no_grad():
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)

# Get top 5 matches
topk = torch.topk(probs, k=5)

print("\nTop 5 matches:\n")
for idx, score in zip(topk.indices[0], topk.values[0]):
    print(f"{descriptions[idx]} --> {score.item():.4f}")